In [ ]:
import numpy as np
from dataclasses import dataclass

In [ ]:
@dataclass
class SinkhornNewtonConfig:
    epsilon : float = 1e-3 #target (final epsilon)
    max_iter : int = 100 
    tol : float = 1e-10
    inner_tol : float = 1e-12
    inner_maxiter : int = 1000

    use_preconditionner : bool = True

    verbose : bool = False
    log_every : int = 1

    use_annealing : bool = True
    epsilon_start : float = 1.0
    epsilon_decay : float = 0.9
    anneal_tol : float = 1e-6
    anneal_max_iter : int = 100 #Inner loop max iteration 
    min_epsilon : float = 1e-8


In [ ]:
@dataclass   
class SinkhornNewtonResult:
    P : np.array
    f : np.array
    g : np.array
    iterations : int
    converged : bool
    history : list[dict[str, Any]] = field(default_factory = list)

In [ ]:
def compute_cost_matrix(x, y, cost_fn):
    x = np.atleast_2d(x)
    y = np.atleast_2d(y)
    C = np.asarray(cost_fn(x,y))
    
    return C

In [ ]:
class SinkhornNewton:
    def __init__(self, laplacian_solver, config = None):
        self.solver = laplacian_solver
        self.config = config or SinkhornNewtonConfig()


    def build_newton_system(self, P, epsilon):  #Builds the jacobian associated to the transport plan
        n, m = P.shape
        a_k = P.sum(axis = 1)
        b_k = P.sum(axis = 0)

        J = np.zeros((n+m, n+m))
        J[:n, :n] = np.diag(a_k) #Top left block
        J[:n, n:] = P            #Top right block 
        J[n:, :n] = P.T          #Bottom left block 
        J[n:, n:] = np.diag(b_k) #Bottom right block 

        J = J / epsilon

        return J, a_k, b_k

            

    def solve_newton_system(self, J, rhs, delta_f0 = None, delta_g0 = None):
        n = J.shape[0] // 2
        m = J.shape[0] - n

        ones_n = np.ones(n)
        ones_m = np.ones(m)
        kernel = np.concatenate([ones_n, -ones_m])

        kernel_norm_sq = n + m
        rhs_proj = rhs - (np.dot(kernel, rhs) / kernel_norm_sq) * kernel 

        x0 = None
        if delta_f0 is not None and delta_g0 is not None:
            x0 = np.concatenate([delta_f0, delta_g0])

        x, info = self.solver.solve(J, rhs_proj, x0 = x0, tol = self.config.inner_tol, maxiter = self.config.inner_maxiter)

        delta_f = x[:n]
        delta_g = x[n:]

        mean_f = np.mean(delta_f)
        mean_g = np.mean(delta_g)
        shift = (mean_f - mean_g) / 2
        delta_f = delta_f - shift
        delta_g = delta_g + shift

        return delta_f, delta_g, info

    def _build_epsilon_schedule(self, eps_target):
        if not self.config.use_annealing:
            return [eps_target]

        eps_start = self.config.epsilon_start

        if eps_start <= eps_target:
            return eps_target

        schedule = []
        eps = eps_start
        while eps > eps_target: * (1 + 1e-12):
            schedule.append(float(eps))
            eps *= self.config.epsilon_decay
            if eps < self.config.min_epsilon:
                break
        schedule.append(float(eps_target))
        return schedule 

    def _solve_single_epsilon(self, a, b, C, eps, P_init, f_init, g_init, tol, max_iter):
        n, m = C.shape
        K = np.exp(-C/eps)

        P = P_init.copy()
        f = f_init.copy()
        g = g_init.copy()

        history_local = []
        converged = False
        k = 0

        for k in range(max_iter):
            a_k = P.sum(axis = 1)
            b_k = P.sum(axis = 0)

            viol_a = float(np.max(np.abs(a_k - a)))
            viol_b = float(np.max(np.abs(b_k - b)))
            maxi_viol = np.max(viol_a, viol_b)

            if self.config.verbose and (k % self.config.log_every == 0):
                print(f"    iter {k}: max_viol = {max_viol:.3e}")

            if max_viol < tol:
                converged = True
                break

            J, a_k, b_k = self.build_newton_system(P, eps)
            rhs = np.concatenate([a_k - a, b_k - b])
            delta_f, delta_g, solver_info = self.solve_newton_system(J, rhs)

            exp_df = np.exp(-delta_f / eps)
            exp_dg = np.exp(-delta_g / eps)
            P = np.diag(exp_df) @ P @ np.diag(exp_dg)

            f = f + delta_f
            g = g + delta_g

            history_local.append({
                'iteration': k,
                'viol_a': viol_a,
                'viol_b': viol_b,
                'max_viol': max_viol,
                'solver_info': solver_info,
                'norm_delta_f': float(np.linalg.norm(delta_f)),
                'norm_delta_g': float(np.linalg.norm(delta_g)),
            })

        return P, f, g, converged, history_local, k + 1

    @staticmethod
    def _build_P_from_potentials(f, g, C, eps):
        K = np.exp(-C / eps)
        exp_f = np.exp(-f / eps)
        exp_g = np.exp(-g / eps)
        return np.diag(exp_f) @ K @ np.diag(exp_g)

        
    def solve(self, a, b, C, epsilon = None, f0 = None, g0 = None):
        n, m = C.shape
        eps_target = epsilon if epsilon is not None else self.config.epsilon

        a = a / a.sum()
        b = b / b.sum()

        eps_schedule = self.build_epsilon_schedule(eps_target)

        if self.config.verbose:
            print(f"Annealing schedule : {len(eps_schedule)} steps, "
                  f"from {eps_schedule[0]:.3e} to {eps_schedule[-1]:.3e}")

        eps_first = eps_schedule[0]
        if f0 is not None:
            f = np.asarray(f0, dtype = float).copy()
        else:
            f = np.zeros(n)

        if g0 is not None:
            g = np.asarray(g0, dtype = float).copy()
        else:
            g = np.zeros(m)

        P = self._build_P_from_potentials(f, g, C, eps_first)
        full_history = []
        global_iter_count = 0
        all_converged = True

        for stage_idx, eps in enumerate(eps_schedule):
            is_last_stage = (stage_idx == len(eps_schedule) - 1)

            stage_tol = self.config.tol if is_last_stage else self.config.anneal_tol
            stage_max_iter = (self.config.max_iter if is_last_stage else self.config.anneal_max_iter)

            if self.config.verbose:
                print(f"\n=== Stage {stage_idx + 1}/{len(eps_schedule)} : "
                    f"eps = {eps:.3e}, tol = {stage_tol:.1e} ===")

            P, f, g, stage_converged, stage_history, n_iter = self._solve_single_epsilon(
                a, b, C, eps, P, f, g, tol=stage_tol, max_iter=stage_max_iter)

            if not stage_converged:
                if is_last_stage:
                    all_converged = False
                    if self.config.verbose:
                        print(f"Final stage did NOT converge in {n_iter} iterations")
                else:
                    warnings.warn(
                    f"Annealing stage {stage_idx + 1}/{len(eps_schedule)} "
                    f"(eps={eps:.3e}) did NOT converge after {n_iter} iterations. "
                    f"Subsequent stages may start from a suboptimal P."
                )

            for entry in stage_history:
                entry['stage'] = stage_idx
                entry['epsilon'] = float(eps)
                entry['global_iteration'] = global_iter_count
                global_iter_count += 1
                full_history.append(entry)


            if self.config.verbose:
                status = "converged" if stage_converged else "NOT converged"
                print(f"  -> {status} in {n_iter} iterations")

        return SinkhornNewtonResult(P=P, f=f, g=g, iterations = global_iter_count, converged = all_converged, history = full_history)
        